# Setup

In [1]:
%%capture
# 1. Install dependencies
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai pandas
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [ ]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent
from pydantic import BaseModel

MODEL = OpenAIChatModel(
    'openai/gpt-4o-mini',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

# Tools

A **plain LLM call** has no memory of the world after its training cutoff and no tools. An **agent** has a loop: it can call tools, see the results, and reason over them.

In [6]:
# (a) Plain agent — no tools
from pydantic_ai import Agent

plain_agent = Agent(MODEL)
result = plain_agent.run_sync('What is today\'s date?')
print(result.output)

Today's date is October 5, 2023.


In [7]:
# (b) Agent with one tool
from datetime import date

def get_today() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return date.today().isoformat()

tool_agent = Agent(MODEL, tools=[get_today])

result = tool_agent.run_sync('What is today\'s date?')
print(result.output)

Today's date is July 2, 2026.


In [8]:
# Tool name, arguments, and docstring are visible to the agent, but not the implementation
from pprint import pprint
for toolset in tool_agent.toolsets:
    for name, tool in toolset.tools.items():
        print(name)
        pprint(tool.function_schema)

get_today
FunctionSchema(function=<function get_today at 0x10f3604a0>,
               name='get_today',
               description="Return today's date in YYYY-MM-DD format.",
               validator=<pydantic.plugin._schema_validator.PluggableSchemaValidator object at 0x10f485740>,
               json_schema={'additionalProperties': False,
                            'properties': {},
                            'type': 'object'},
               takes_ctx=False,
               is_async=False,
               single_arg_name=None,
               positional_fields=[],
               var_positional_field=None,
               return_schema={'type': 'string'})


**What just happened?**

The first agent had no way to know the current date. The second agent had a tool — `get_today()` — and the LLM decided to call it.

Critically, the LLM doesn't execute the function itself. It returns a *tool-call request*, the framework runs the function, the result is appended to the conversation, and the LLM then produces its final answer.


# Trace 

In [9]:
print(f'Final answer: {result.output}\n')
print('Trace (each ModelMessage in the conversation):')

def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)[:200]
            print(f'    └─ {kind}: {snippet}')

pretty_print_trace(result)

Final answer: Today's date is July 2, 2026.

Trace (each ModelMessage in the conversation):

[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content="What is today's date?", timestamp=datetime.datetime(2026, 7, 2, 12, 48, 34, 978772, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='get_today', args='{}', tool_call_id='call_q7ATpotwKoLHmhCofie30PIZ')

[2] ModelRequest
    └─ ToolReturnPart: ToolReturnPart(tool_name='get_today', content='2026-07-02', tool_call_id='call_q7ATpotwKoLHmhCofie30PIZ', timestamp=datetime.datetime(2026, 7, 2, 12, 48, 35, 951379, tzinfo=datetime.timezone.utc))

[3] ModelResponse
    └─ TextPart: TextPart(content="Today's date is July 2, 2026.")


# Exercise

In [10]:
# Add a tool to the tool_agent called count_words that returns the number of whitespace-separated words in a piece of text.
tool_agent = Agent(MODEL)


In [11]:
# Test
text_to_count = ""
result = tool_agent.run_sync(f"how many words are in this text? : {text_to_count}")
pretty_print_trace(result)


[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content='how many words are in this text? : ', timestamp=datetime.datetime(2026, 7, 2, 12, 48, 45, 370991, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ TextPart: TextPart(content="It appears that there is no text provided for me to count the words. Please provide the text you'd like me to analyze, and I'll count the words for you.")


In [ ]:
# BONUS: Add a tool that will count the number of the letter 'r' in a word

In [12]:
# Test
result = tool_agent.run_sync(f"how many r's are in the word strawberry?")
pretty_print_trace(result)


[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content="how many r's are in the word strawberry?", timestamp=datetime.datetime(2026, 7, 2, 12, 48, 50, 434328, tzinfo=datetime.timezone.utc))

[1] ModelResponse
    └─ TextPart: TextPart(content='The word "strawberry" contains 2 "r\'s."')


# New stuff from now on
The plan:
make them write small GC-content function and show that its much better than LLM
create 2 fake data csvs and have them write a function to get the summary + try it out (showing multiple tool calls to answer the user question)
then have them write add_gc_content_column (re-using their old single-sequence GC function through apply) -> try it out (showing agent can reason about what to call for different questions but with the same tools)

In [ ]:
# count k-mers
# count GC
# access some local DB?
# parse csv and get some statistic (e.g. how many are there), unique names of a column etc (do 2+ csvs to show repeated tool usage)?
# list working dir
# write a text report
# make a plot
# Fetch PubMed metadata by PMID or Search PubMed for recent papers
# bash tool? 

In [43]:
import random
dna_sequence = "".join(random.choices("ATGC", weights=[0.15,0.15,0.35,0.35], k=100))
Agent(MODEL).run_sync(f"Whats the GC content of {dna_sequence}?")

AgentRunResult(output="To calculate the GC content of the given DNA sequence, you count the number of guanine (G) and cytosine (C) nucleotides and divide by the total number of nucleotides in the sequence.\n\nHere's a step-by-step breakdown:\n\n1. Count the total number of bases in the sequence: \n   The provided sequence has a total of 75 bases.\n\n2. Count the number of G and C bases:\n   - C count = 20\n   - G count = 21\n\n3. Calculate the GC content:\n   \\[\n   \\text{GC content} = \\frac{(\\text{G count} + \\text{C count})}{\\text{Total bases}} \\times 100\n   \\]\n   \\[\n   \\text{GC content} = \\frac{(21 + 20)}{75} \\times 100 = \\frac{41}{75} \\times 100 \\approx 54.67\\%\n   \\]\n\nTherefore, the GC content of the given DNA sequence is approximately **54.67%**.")

In [44]:
# Create an agent with a tool to calculate GC content

def get_gc_content(sequence: str):
    return 100 * ((sequence.count('C') + sequence.count('G')) / len(sequence))

gc_agent = Agent(MODEL, tools=[get_gc_content])
result = gc_agent.run_sync(f"whats the GC content of {dna_sequence}")
pretty_print_trace(result)


[0] ModelRequest
    └─ UserPromptPart: UserPromptPart(content='whats the GC content of CCACCGCATCATACGGCCACCGGGGGGCATCGCTGGCTCGGGATCCAAGTGGCTGGCCTTCCTGTATGCCCCTCCACTGGCGCCCTGGCCGTGGAGCGTA', timestamp=datetime.datetime(2026, 7, 2, 14, 0, 24

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='get_gc_content', args='{"sequence":"CCACCGCATCATACGGCCACCGGGGGGCATCGCTGGCTCGGGATCCAAGTGGCTGGCCTTCCTGTATGCCCCTCCACTGGCGCCCTGGCCGTGGAGCGTA"}', tool_call_id='call_2uO1peVp8Z2AlLli

[2] ModelRequest
    └─ ToolReturnPart: ToolReturnPart(tool_name='get_gc_content', content=69.0, tool_call_id='call_2uO1peVp8Z2AlLlikZBU5nmi', timestamp=datetime.datetime(2026, 7, 2, 14, 0, 25, 986024, tzinfo=datetime.timezone.utc))

[3] ModelResponse
    └─ TextPart: TextPart(content='The GC content of the given sequence is 69.0%.')


In [ ]:
# create a sample csv with random genomic sequences
import random
import pandas as pd



def make_an_experiment_csv(num_of_samples, csv_path, seed=42):
    rng = random.Random(seed)
    sequence_lengths = [rng.randint(100, 1000) for _ in range(num_of_samples)]
    pd.DataFrame({
        "sequence": [
            "".join(rng.choices("ATCG", k=sequence_lengths[i]))
            for i in range(num_of_samples)
        ],
        "sequence_len": sequence_lengths,
        "score": [100/(rng.random()*sequence_lengths[i]) for i in range(num_of_samples)]
    }).to_csv(csv_path, index=False)

make_an_experiment_csv(2000, "experiment_1.csv", 42)
make_an_experiment_csv(1000, "experiment_2.csv", 69)

# Tools can return anything that Pydantic can serialize to JSON https://pydantic.dev/docs/ai/tools-toolsets/tools/#function-tool-output


In [31]:
get_csv_stats('experiment_1_with_gc.csv')

{'sequence_len': {'count': 2000.0,
  'mean': 556.3015,
  'std': 259.6456711270525,
  'min': 100.0,
  '25%': 338.0,
  '50%': 558.5,
  '75%': 781.0,
  'max': 1000.0},
 'score': {'count': 2000.0,
  'mean': 2.0168682804324036,
  'std': 10.41287424176001,
  'min': 0.1019963954792324,
  '25%': 0.24634521436865936,
  '50%': 0.46264719729787845,
  '75%': 1.010683529468561,
  'max': 196.4311687294428},
 'GC_content': {'count': 2000.0,
  'mean': 50.002114826103394,
  'std': 2.534010820577185,
  'min': 38.72340425531915,
  '25%': 48.55620026525199,
  '50%': 50.0,
  '75%': 51.550618859965596,
  'max': 60.18518518518518}}

In [ ]:
# we want to keep tools generic and efficient (e.g. running compute_gc on 1000s of sequences = 1000s of tool calls)
def add_gc_content_column(csv_to_read: str, sequence_column: str, new_csv_path: str): #TODO types needed otherwise pydantic cries
    df = pd.read_csv(csv_to_read)
    df["GC_content"] = df[sequence_column].apply(lambda x: 100* ((x.count("G") + x.count("C")) / len(x)))
    df.to_csv(new_csv_path, index=False)

def get_csv_stats(csv_path: str):
    return pd.read_csv(csv_path).describe().to_dict() # tool outputs need to be JSON-serializable

agent = Agent(MODEL, tools=[
    add_gc_content_column, 
    get_csv_stats
    ]
)
pprint(agent.run_sync(f"You have access to csvs: [experiment_1.csv, experiment_2.csv]. Which one has the highest GC content sequence?"))

AgentRunResult(output='The GC content statistics for both CSV files are as '
                      'follows:\n'
                      '\n'
                      '1. **experiment_1.csv**:\n'
                      '   - Average GC Content: 50.00%\n'
                      '   - Maximum GC Content: 60.19%\n'
                      '\n'
                      '2. **experiment_2.csv**:\n'
                      '   - Average GC Content: 50.03%\n'
                      '   - Maximum GC Content: 58.95%\n'
                      '\n'
                      'Based on the maximum GC content, **experiment_1.csv** '
                      'has the highest GC content sequence with a maximum of '
                      '**60.19%**.')


In [36]:
pprint(agent.run_sync(f"You have access to csvs: [experiment_1.csv, experiment_2.csv]. Which one has more sequences?"))


AgentRunResult(output='The statistics for the sequences in the CSV files are '
                      'as follows:\n'
                      '\n'
                      '- **experiment_1.csv**: \n'
                      '  - Number of sequences: 2000\n'
                      '\n'
                      '- **experiment_2.csv**: \n'
                      '  - Number of sequences: 1000\n'
                      '\n'
                      'Therefore, **experiment_1.csv** has more sequences.')


In [ ]:
#TODO this is a tool we might want to eventually use maybe in multi-agents? maybe even here? IDK yet

import requests

#TODO these are only from today?
def search_pubmed_abstracts(query: str, top_k: int = 3) -> str:
    """Search PubMed and return abstracts for the top_k results."""
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    ids = requests.get(
        f"{base}/esearch.fcgi",
        params={"db": "pubmed", "term": query, "retmode": "json", "retmax": top_k},
    ).json()["esearchresult"]["idlist"]

    if not ids:
        return "No PubMed results found."

    return requests.get(
        f"{base}/efetch.fcgi",
        params={
            "db": "pubmed",
            "id": ",".join(ids),
            "rettype": "abstract",
            "retmode": "text",
        },
    ).text

res = search_pubmed_abstracts("machine learning")

In [37]:
# pprint(res)

In [35]:
tool_agent = Agent(MODEL, tools=[search_pubmed_abstracts])
pprint(tool_agent.run_sync("Summarize the top 3 papers for ML, give me their ids also"))

AgentRunResult(output="Here's a summary of the top 3 papers related to machine "
                      'learning:\n'
                      '\n'
                      '---\n'
                      '\n'
                      '## **Paper 1: Vibe Coding in Neurosurgery**\n'
                      '**PMID: 42389900**\n'
                      '- **Title:** Vibe Coding in Neurosurgery: Bridging the '
                      'Gap Between Clinical Needs and Digital Innovation\n'
                      '- **Summary:** This narrative review examines "vibe '
                      'coding" (generating software through natural language '
                      'instructions to AI models) and its applications in '
                      'neurosurgery. The paper identifies three practical '
                      'domains: (1) research data collection and patient '
                      'classification, (2) clinical workflow optimization '
                      'including documentation and follow-up automati